In [5]:
import requests
import os
import re
import gzip
import shutil
import pandas as pd
from collections import defaultdict

In [7]:
BASE_URL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
# rename output directory to whatever you want
outdir = "stormevents"
os.makedirs(outdir, exist_ok=True)

In [2]:
# this gets details, fatalities, and locations files. details is probably the main thing you want, but might as well grab all.
html = requests.get(BASE_URL).text

pattern = r"StormEvents_(details|fatalities|locations)-ftp_v1.0_d(\d{4})_c(\d{8})\.csv\.gz"
matches = re.findall(pattern, html)

latest_files = defaultdict(dict)

for ftype, year, catdate in matches:
    key = (year, ftype)
    filename = f"StormEvents_{ftype}-ftp_v1.0_d{year}_c{catdate}.csv.gz"
    current = latest_files[year].get(ftype)

    if not current or catdate > re.search(r"c(\d+)", current).group(1):
        latest_files[year][ftype] = filename

for year, files in latest_files.items():
    for ftype, filename in files.items():
        url = BASE_URL + filename
        path = os.path.join(outdir, filename)

        print(f"Downloading {filename}")
        try:
            r = requests.get(url, stream=True)
            if r.ok:
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
            else:
                print(f"Failed: {filename} ({r.status_code})")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")


ERROR! Session/line number was not unique in database. History logging moved to new session 604


In [3]:
dir_name = outdir

from pathlib import Path

# unzip archives and replace them with the extracted csvs
def gz_extract(directory):
    # adjusted from https://gist.github.com/kstreepy/a9800804c21367d5a8bde692318a18f5

    for gz_file in Path(directory).glob("*.gz"):
        csv_file = gz_file.with_suffix("")  # removes .gz
        if csv_file.exists():
            continue

        with gzip.open(gz_file, "rb") as f_in, open(csv_file, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_file.unlink()

        
gz_extract(dir_name)

In [8]:
# convert the files to dataframes, one df per type of file
typed_dfs = defaultdict(list)
for filename in os.listdir(outdir):
    filepath = os.path.join(outdir, filename)
    if not filename.endswith(".csv"):
        continue

    if "details" in filename:
        ftype = "details"
    elif "fatalities" in filename:
        ftype = "fatalities"
    elif "locations" in filename:
        ftype = "locations"
    else:
        continue

    df = pd.read_csv(filepath, low_memory=False)
    typed_dfs[ftype].append(df)

In [9]:
# combine all the years
details_df = pd.concat(typed_dfs["details"], ignore_index=True)
fatalities_df = pd.concat(typed_dfs["fatalities"], ignore_index=True)
locations_df = pd.concat(typed_dfs["locations"], ignore_index=True)

C:\Users\micah\AppData\Local\Temp\ipykernel_18472\4021630058.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  locations_df = pd.concat(typed_dfs["locations"], ignore_index=True)


In [10]:
# filter it down to tornadoes only
tornado_details_df = details_df[details_df.EVENT_TYPE == "Tornado"]
tornado_details_df

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,195004,28,1445,195004,28,1445,NaN,10096222,OKLAHOMA,40.0,...,0.0,NaN,NaN,35.1200,-99.2000,35.1700,-99.2000,NaN,NaN,PUB
1,195004,29,1530,195004,29,1530,NaN,10120412,TEXAS,48.0,...,0.0,NaN,NaN,31.9000,-98.6000,31.7300,-98.6000,NaN,NaN,PUB
2,195007,5,1800,195007,5,1800,NaN,10104927,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,40.5800,-75.7000,40.6500,-75.4700,NaN,NaN,PUB
3,195007,5,1830,195007,5,1830,NaN,10104928,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,40.6000,-76.7500,NaN,NaN,NaN,NaN,PUB
4,195007,24,1440,195007,24,1440,NaN,10104929,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,41.6300,-79.6800,NaN,NaN,NaN,NaN,PUB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940809,202407,16,1548,202407,16,1552,193998.0,1199967,NEW YORK,36.0,...,5.0,NW,HARRISBURG,43.4452,-74.1713,43.4707,-74.1368,The second of three days of hot and humid cond...,"A National Weather Service storm survey, with ...",CSV
1940816,202407,6,1632,202407,6,1634,192857.0,1198798,NEBRASKA,31.0,...,3.0,SE,SUTTON,40.5544,-97.8303,40.5700,-97.8200,"A broad, upper trough was over most of the cou...",A QLCS tornado affected portions of eastern Cl...,CSV
1940839,202407,6,1536,202407,6,1540,192857.0,1198800,NEBRASKA,31.0,...,3.0,ESE,ROSELAND,40.4647,-98.5494,40.4512,-98.5081,"A broad, upper trough was over most of the cou...",A small tornado developed within a broader are...,CSV
1940851,202407,16,1622,202407,16,1629,193998.0,1200680,NEW YORK,36.0,...,3.0,N,BOLTON LNDG,43.5655,-73.8502,43.6078,-73.6667,The second of three days of hot and humid cond...,A National Weather Service storm survey determ...,CSV


In [19]:
tornado_details_df.DAMAGE_PROPERTY.value_counts()

DAMAGE_PROPERTY
0K         9319
0.00K      9161
25K        8951
250K       6465
2.5K       4879
           ... 
212K          1
670K          1
129.00K       1
9.75K         1
10.40K        1
Name: count, Length: 917, dtype: int64

In [11]:
# need this to get the other 2 df filtered too
tornado_event_ids = tornado_details_df["EVENT_ID"].unique()

In [12]:
tornado_fatalities_df = tornado_fatalities = fatalities_df[fatalities_df["EVENT_ID"].isin(tornado_event_ids)]
tornado_fatalities_df

,FAT_YEARMONTH,FAT_DAY,FAT_TIME,FATALITY_ID,EVENT_ID,FATALITY_TYPE,FATALITY_DATE,FATALITY_AGE,FATALITY_SEX,FATALITY_LOCATION,EVENT_YEARMONTH
0,195001,13,525,1005198,9981922,D,01/13/1950 05:25:00,NaN,NaN,NaN,195001.0
1,195002,12,1200,1005199,10049525,D,02/12/1950 12:00:00,NaN,NaN,NaN,195002.0
2,195002,11,1350,1005200,10120403,D,02/11/1950 13:50:00,NaN,NaN,NaN,195002.0
3,195002,12,30,1005201,10120406,D,02/12/1950 00:30:00,NaN,NaN,NaN,195002.0
4,195002,12,1200,1005202,10120410,D,02/12/1950 12:00:00,NaN,NaN,NaN,195002.0
...,...,...,...,...,...,...,...,...,...,...,...
24133,202404,27,0,52835,1175819,D,04/27/2024 00:00:00,35.0,M,Mobile/Trailer Home,202404.0
24134,202404,27,0,52836,1175819,D,04/27/2024 00:00:00,0.0,F,Mobile/Trailer Home,202404.0
24199,202403,14,0,56102,1166418,D,03/14/2024 00:00:00,69.0,M,Unknown,202403.0
24200,202403,14,0,56100,1166418,D,03/14/2024 00:00:00,70.0,F,Mobile/Trailer Home,202403.0


In [13]:
tornado_locations_df = locations_df[locations_df["EVENT_ID"].isin(tornado_event_ids)]
tornado_locations_df

,YEARMONTH,EPISODE_ID,EVENT_ID,LOCATION_INDEX,RANGE,AZIMUTH,LOCATION,LATITUDE,LONGITUDE,LAT2,LON2
0,197206,990000001,990000001,1,NaN,NaN,LABELLE,26.7700,-81.4800,2677.0,-8148.0
1,197206,990000001,990000001,2,NaN,NaN,LABELLE,26.7800,-81.4800,2678.0,-8148.0
7,199603,2030063,5548857,1,2.00,S,MIDWAY,36.3500,-92.4700,3621.0,9228.0
27,199603,2030070,5548864,2,3.00,WSW,SIDNEY,35.9800,-91.7200,3559.0,9143.0
29,199603,2030072,5548866,2,NaN,NaN,PEARCY,34.4300,-93.2800,3426.0,9317.0
...,...,...,...,...,...,...,...,...,...,...,...
1727072,202408,195921,1211614,1,3.07,SW,BUFFALO,42.8894,-78.8940,4253364.0,7853640.0
1727073,202408,195921,1211614,2,2.07,SSW,BUFFALO,42.8930,-78.8675,4253580.0,7852050.0
1727776,202408,194220,1201365,1,0.86,NW,NESHANNOCK,41.2300,-80.4100,4113800.0,8024600.0
1728195,202408,196338,1215115,1,0.84,E,SPIRIT LAKE LAKE,43.4713,-95.1034,4328278.0,956204.0


In [20]:
# crops were making the file saving mess up, so instead of '5K' it's 5000, etc.
def parse_damage(val):
    if pd.isna(val) or not str(val).strip():
        return None

    val = str(val).strip().upper()

    try:
        # Handle 'K', 'M', 'B' with and without decimals
        if val.endswith("K") and len(val) > 1:
            return float(val[:-1]) * 1_000
        elif val.endswith("M") and len(val) > 1:
            return float(val[:-1]) * 1_000_000
        elif val.endswith("B") and len(val) > 1:
            return float(val[:-1]) * 1_000_000_000
        else:
            return float(val)
    except ValueError:
        return None

tornado_details_df["DAMAGE_CROPS"] = tornado_details_df["DAMAGE_CROPS"].apply(parse_damage)
tornado_details_df["DAMAGE_PROPERTY"] = tornado_details_df["DAMAGE_PROPERTY"].apply(parse_damage)

C:\Users\micah\AppData\Local\Temp\ipykernel_18472\3395282788.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tornado_details_df["DAMAGE_CROPS"] = tornado_details_df["DAMAGE_CROPS"].apply(parse_damage)
C:\Users\micah\AppData\Local\Temp\ipykernel_18472\3395282788.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tornado_details_df["DAMAGE_PROPERTY"] = tornado_details_df["DAMAGE_PROPERTY"].apply(parse_damage)


In [21]:
# save to parquet files
tornado_details_df.to_parquet('processed_files/tornado_details.parquet')
tornado_fatalities_df.to_parquet('processed_files/tornado_fatalities.parquet')
tornado_locations_df.to_parquet('processed_files/tornado_locations.parquet')

In [24]:
tornado_details_df_cleaned = tornado_details_df.dropna(axis=1, how='all')
tornado_details_df_cleaned

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,195004,28,1445,195004,28,1445,NaN,10096222,OKLAHOMA,40.0,...,0.0,NaN,NaN,35.1200,-99.2000,35.1700,-99.2000,NaN,NaN,PUB
1,195004,29,1530,195004,29,1530,NaN,10120412,TEXAS,48.0,...,0.0,NaN,NaN,31.9000,-98.6000,31.7300,-98.6000,NaN,NaN,PUB
2,195007,5,1800,195007,5,1800,NaN,10104927,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,40.5800,-75.7000,40.6500,-75.4700,NaN,NaN,PUB
3,195007,5,1830,195007,5,1830,NaN,10104928,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,40.6000,-76.7500,NaN,NaN,NaN,NaN,PUB
4,195007,24,1440,195007,24,1440,NaN,10104929,PENNSYLVANIA,42.0,...,0.0,NaN,NaN,41.6300,-79.6800,NaN,NaN,NaN,NaN,PUB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940809,202407,16,1548,202407,16,1552,193998.0,1199967,NEW YORK,36.0,...,5.0,NW,HARRISBURG,43.4452,-74.1713,43.4707,-74.1368,The second of three days of hot and humid cond...,"A National Weather Service storm survey, with ...",CSV
1940816,202407,6,1632,202407,6,1634,192857.0,1198798,NEBRASKA,31.0,...,3.0,SE,SUTTON,40.5544,-97.8303,40.5700,-97.8200,"A broad, upper trough was over most of the cou...",A QLCS tornado affected portions of eastern Cl...,CSV
1940839,202407,6,1536,202407,6,1540,192857.0,1198800,NEBRASKA,31.0,...,3.0,ESE,ROSELAND,40.4647,-98.5494,40.4512,-98.5081,"A broad, upper trough was over most of the cou...",A small tornado developed within a broader are...,CSV
1940851,202407,16,1622,202407,16,1629,193998.0,1200680,NEW YORK,36.0,...,3.0,N,BOLTON LNDG,43.5655,-73.8502,43.6078,-73.6667,The second of three days of hot and humid cond...,A National Weather Service storm survey determ...,CSV


In [23]:
tornado_details_df.columns[tornado_details_df.isna().all()].tolist()

['FLOOD_CAUSE', 'CATEGORY']

In [25]:
import nbformat

nb = nbformat.read("storm_events.ipynb", as_version=4)

# Set missing execution_count to null
for cell in nb.cells:
    if cell.cell_type == "code" and "execution_count" not in cell:
        cell["execution_count"] = None
    if "outputs" not in cell:
        cell["outputs"] = []

nbformat.write(nb, "storm_events.ipynb")
